### **AMq-TP-Co20**

### **Aprendizaje de Máquina**

### **Co20**

### **TP**

##### **Integrantes:**

* Natalia Beatriz-Diaz
* Francisco Antonio Cofré Villalón
* José Luis Diaz
* José Manuel Aviani


---

#### Dataset: Kaggle - Red Wine Quality

https://www.kaggle.com/datasets/uciml/red-wine-quality-cortez-et-al-2009

Vamos a estimar el *target "quality"*.

Para ello vamos a implementar diferentes modelos diferentes que en principio son adecuados para este tipo de problemas.

Analizaremos las ventajas y desventajas de cada uno y finalmente seleccionaremos el mejor según el criterio más adecuado.

---

In [ ]:
%run 01_Random_Forest.ipynb
%run 02_K_Nearest_Neighbors.ipynb
%run 03_Logistic_Regression.ipynb

In [ ]:
import importlib
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
import random
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.utils.class_weight import compute_class_weight

import Gradient_Boosting_Model as gbm
import XGBoost_Model as xgbm

TODO: Completar requirements.txt

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
EJECUTAR_RANDOM_FOREST = False
EJECUTAR_K_NEAREST_NEIGHBORS = False
EJECUTAR_LOGISTIC_REGRESSION = False
EJECUTAR_GRADIENT_BOOSTING = False
EJECUTAR_XGBOOST = False

Descargamos el dataset:

In [ ]:
df = pd.read_csv("./dataset/winequality-red.csv")

Analizamos los datos:

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
def graficar_distribucion_target():
  # Calcular frecuencias absolutas y relativas
  quality_counts = df["quality"].value_counts().sort_index()
  quality_perc = (quality_counts / quality_counts.sum()) * 100

  # Gráfico de barras con dos ejes (conteo absoluto y %)
  fig, ax1 = plt.subplots(figsize=(8,5))

  # Barras (conteo absoluto)
  bars = ax1.bar(quality_counts.index, quality_counts.values, color="skyblue")
  ax1.set_xlabel("Calidad del vino (quality)")
  ax1.set_ylabel("Número de muestras", color="blue")
  ax1.tick_params(axis="y", labelcolor="blue")

  # Segundo eje Y para porcentajes
  ax2 = ax1.twinx()
  ax2.plot(quality_counts.index, quality_perc.values, color="red", marker="o", linewidth=2)
  ax2.set_ylabel("Porcentaje (%)", color="red")
  ax2.tick_params(axis="y", labelcolor="red")

  # Títulos y ajustes
  plt.title("Distribución del target 'quality' (conteo absoluto y %)")
  plt.xticks(quality_counts.index)

  # Mostrar valores encima de cada barra
  for bar, perc in zip(bars, quality_perc.values):
      height = bar.get_height()
      ax1.text(bar.get_x() + bar.get_width()/2, height+10,
              f"{int(height)}\n({perc:.1f}%)",
              ha="center", va="bottom", fontsize=8)

  plt.tight_layout()
  plt.show()


graficar_distribucion_target()

Revisión del dataset:

* Filas: 1.599

* Columnas: 12 (11 predictoras numéricas continuas + target quality).

* Valores faltantes: Ninguno.

* Tipo de datos: Todo numérico (float en features, int en target).

* Target (quality): valores entre 3 y 8, pero con distribución desbalanceada (la mayoría son 5 y 6).

---

### Preprocesamiento:

#### Manejo de variables predictoras:

Como todos los atributos son numéricos, no se necesita one-hot encoding ni transformaciones categóricas.

Normalización y estandarización: No lo aplicamos en este preprocesamiento general. En la sección propia de cada modelo se analizará si son necesarios para el modelo específico.

Outliers: Vamos detectar y tratar outliers extremos, ya que algunos atributos (residual sugar, chlorides, total sulfur dioxide) tienen colas largas que podrían afectar el aprendizaje.

---
### Outliers


In [ ]:
def graficar_outliers():
  features = df.drop('quality', axis=1).columns

  # Crear box plots para identificar valores atípicos
  fig, axes = plt.subplots(nrows=4, ncols=3, figsize=(18, 15))
  axes = axes.flatten()

  for i, col in enumerate(features):
      sns.boxplot(x=df[col], ax=axes[i], color='lightblue')
      axes[i].set_title(f'Box Plot de {col}')

  plt.tight_layout()
  plt.show()


graficar_outliers()

Los box plots confirman visualmente la existencia de numerosos valores atípicos en la mayoría de las variables, especialmente en aquellas con distribuciones asimétricas. La gestión de estos valores atípicos será un tema crucial en la fase de preprocesamiento de datos.

### Correlación

In [ ]:
def graficar_correlacion():
    corr_matrix = df.corr()

    plt.figure(figsize=(12, 10))
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
    plt.title('Mapa de Calor de Correlación de las Características del Vino')
    plt.show()


graficar_correlacion()

#

#### Correlaciones con la calidad

La variable alcohol muestra la correlación positiva más fuerte con calidad (0.48). Esto podria sugerir que, los vinos con mayor contenido de alcohol tienden a recibir puntuaciones de calidad más altas.

La volatile_acidity presenta la correlación negativa más significativa (-0.39). Esto podria sugerir que, al revez que en caso anterior, mayor es la acidez volatil peor es la calificacion que obtiene el vino.

#### Comportamiento de las features comparadas con los Niveles de Calidad
El boxplot a continuacion, compara la distribución de cada feature para cada nivel de calidad.


In [ ]:
def graficar_features_target():
    features = df.drop('quality', axis=1).columns
    fig, axes = plt.subplots(nrows=4, ncols=3, figsize=(18, 15))
    axes = axes.flatten()

    for i, col in enumerate(features):
        sns.boxplot(x='quality', y=col, data=df, ax=axes[i], palette='pastel', hue='quality')
        axes[i].set_title(f'{col} vs. Calidad')

    plt.tight_layout()
    plt.show()

graficar_features_target()


Algunas obvservaciones claves:

- Se observa una clara tendencia ascendente en la mediana del `alcohol` a medida que aumenta la puntuación de calidad.
- La mediana de `volatile_acidity` muestra una tendencia descendente igualmente clara.
- `citric_acid` y `sulphates` tienden a aumentar con la calidad, aunque con mas variabilidad.
- `density` tiende a disminuir ligeramente a medida que aumenta la calidad.


#### Target - Balanceo de clases:

El dataset está desbalanceado: la mayoría son 5 y 6 (ver gráfico de más arriba).

Vamos a aplicar:

* Estratificación en train/test split.

* Balanceo por peso de las clases (class_weight="balanced").

TODO: Balanceo por peso de las clases (class_weight="balanced"). Aplicar en cada modelo. Hay un calculo general de los pesos.

----

#### Split de datos:

Hacemos un split estratificado, de modo que la proporción de clases en *quality* se mantenga en train y test:

In [ ]:
test_size = 0.3

X = df.drop("quality", axis=1)
y = df["quality"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y, 
    test_size=test_size,
    stratify=y
)

In [ ]:
print("Tamaño X_train:", X_train.shape)
print("Tamaño X_test:", X_test.shape)
print("Tamaño y_train:", y_train.shape)
print("Tamaño y_test:", y_test.shape)
print("Distribución en train:\n", y_train.value_counts(normalize=True))
print("Distribución en test:\n", y_test.value_counts(normalize=True))

Calculo el peso de las clases para luego usar en cada modelo:

In [ ]:
classes = np.unique(y_train)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weights = dict(zip(classes, weights))

In [ ]:
classes

In [ ]:
class_weights

---

### Random Forest

TODO: Natalia

In [ ]:
if EJECUTAR_RANDOM_FOREST:
    Random_Forest(X_train.copy(), X_test.copy(), y_train.copy(), y_test.copy(), classes.copy(), class_weights.copy())

---

### K-Nearest Neighbors

TODO: Francisco

In [ ]:
if EJECUTAR_K_NEAREST_NEIGHBORS:
    K_Nearest_Neighbors(X_train.copy(), X_test.copy(), y_train.copy(), y_test.copy(), classes.copy(), class_weights.copy())

---

### Logistic Regression (multiclase)

TODO: José Luis

In [ ]:
if EJECUTAR_LOGISTIC_REGRESSION:
    Logistic_Regression(X_train.copy(), X_test.copy(), y_train.copy(), y_test.copy(), classes.copy(), class_weights.copy())

---

### Gradient Boosting

TODO: José

In [ ]:
if EJECUTAR_GRADIENT_BOOSTING:
  importlib.reload(gbm)
  gradient_boosting_model = gbm.GradientBoostingModel(X_train.copy(), X_test.copy(), y_train.copy(), y_test.copy(), classes.copy(), class_weights.copy())
  gradient_boosting_model.procesar()

---

### XGBoost

TODO: José

In [ ]:
if EJECUTAR_XGBOOST:
    importlib.reload(xgbm)
    xgboost_model = xgbm.XGBoostModel(X, y)
    xgboost_model.procesar()

---

### Conclusiones:

TODO: Conclusiones

---